# Audio Augmentation Testing

This notebook tests the audio augmentation methods from `data_augmentation.py` on the test audio files.

In [ ]:
import os
import sys
import librosa
import numpy as np
import soundfile as sf
from pathlib import Path

# Add the current directory to the path so we can import data_augmentation
sys.path.append('.')

# Import our augmentation functions
from data_augmentation import randomly_eq, apply_compression, add_noise

print("Libraries imported successfully!")

In [ ]:
# Create output folder
output_folder = 'augmentations_test_outputs'
os.makedirs(output_folder, exist_ok=True)

# List test audio files
test_audio_folder = 'test_audio'
audio_files = [f for f in os.listdir(test_audio_folder) if f.endswith('.mp3')]
print(f"Found {len(audio_files)} audio files in {test_audio_folder}:")
for audio_file in audio_files[:5]:  # Show first 5
    print(f"  - {audio_file}")
if len(audio_files) > 5:
    print(f"  ... and {len(audio_files) - 5} more")

In [ ]:
# Define the augmentations to test
augmentations = [
    ('randomly_eq', randomly_eq, {'gain_range': (-3, 3)}),  # Mild EQ changes
    ('apply_compression', apply_compression, {'threshold_db': -15, 'ratio': 3.0}),  # Moderate compression
    ('add_noise', add_noise, {'noise_type': 'pink', 'snr_db': -25})  # Pink noise at -25dB SNR
]

print("Augmentations to test:")
for name, func, args in augmentations:
    print(f"  - {name}: {args}")

In [ ]:
def print_audio_info(file_path, label="Audio"):
    """Print information about an audio file."""
    try:
        y, sr = librosa.load(file_path, sr=None)
        duration = len(y) / sr
        file_size = os.path.getsize(file_path)
        print(f"{label}: {file_path}")
        print(f"  Duration: {duration:.2f}s, Sample Rate: {sr}Hz, Size: {file_size} bytes")
        return y, sr
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None, None

# Test on first audio file
if audio_files:
    test_file = os.path.join(test_audio_folder, audio_files[0])
    print(f"\n=== Testing on {audio_files[0]} ===")

    # Print original file info
    y_orig, sr_orig = print_audio_info(test_file, "Original")

    # Apply each augmentation
    for aug_name, aug_func, aug_args in augmentations:
        output_file = os.path.join(output_folder, f"{Path(audio_files[0]).stem}_{aug_name}.wav")
        print(f"\nApplying {aug_name}...")

        try:
            aug_func(test_file, output_file, aug_args)
            print_audio_info(output_file, f"Augmented ({aug_name})")
        except Exception as e:
            print(f"Error applying {aug_name}: {e}")

print("\n=== Testing complete for first file ===")

In [ ]:
# Apply all augmentations to all files
print(f"\n=== Applying augmentations to all {len(audio_files)} files ===")

total_processed = 0
for audio_file in audio_files:
    input_path = os.path.join(test_audio_folder, audio_file)
    base_name = Path(audio_file).stem

    print(f"\nProcessing {audio_file}...")

    for aug_name, aug_func, aug_args in augmentations:
        output_file = os.path.join(output_folder, f"{base_name}_{aug_name}.wav")

        try:
            aug_func(input_path, output_file, aug_args)
            total_processed += 1
            print(f"  ✓ {aug_name} -> {os.path.basename(output_file)}")
        except Exception as e:
            print(f"  ✗ {aug_name} failed: {e}")

print(f"\n=== Processing complete ===")
print(f"Total augmentations applied: {total_processed}")
print(f"Output folder: {output_folder}")
print(f"Files created: {len(os.listdir(output_folder))}")

In [ ]:
# List all output files and their information
print("
=== Output Files Summary ===")
output_files = sorted(os.listdir(output_folder))
for output_file in output_files:
    file_path = os.path.join(output_folder, output_file)
    print_audio_info(file_path, output_file)

print(f"\nTotal output files: {len(output_files)}")
print("All augmentations completed successfully!")